In [7]:
import os
from dotenv import load_dotenv
import pandas as pd
import time
import praw
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from tqdm import tqdm

In [9]:
df = pd.read_csv("Movies.csv")

In [11]:
load_dotenv()

True

In [13]:
client_id = os.getenv("reddit_movie_client_id")
print("API key loaded:", client_id is not None)

API key loaded: True


In [19]:
secret_id = os.getenv("reddit_movie_secret_id")
print("API key loaded:", secret_id is not None)

API key loaded: True


In [21]:
reddit = praw.Reddit(client_id = client_id, client_secret = secret_id, user_agent = 'movieScrapeCHUI')

In [23]:
analyzer = SentimentIntensityAnalyzer()

In [25]:
sentiment_scores = []

In [27]:
for title in tqdm(df["title"]):
    try:
        query = f'title:"{title}"'
        posts = reddit.subreddit("movies").search(query, limit=5)

        weighted_scores = []
        total_weight = 0

        for post in posts:
            post_text = (post.title or "") + " " + (post.selftext or "")
            post_score = analyzer.polarity_scores(post_text)["compound"]
            weight = max(post.score, 1)
            weighted_scores.append(post_score * weight)
            total_weight += weight

            post.comments.replace_more(limit=0)
            for comment in post.comments.list()[:20]:  # Top 20 comments
                if comment.body:
                    comment_score = analyzer.polarity_scores(comment.body)["compound"]
                    comment_weight = max(comment.score, 1)
                    weighted_scores.append(comment_score * comment_weight)
                    total_weight += comment_weight

        if total_weight > 0:
            avg_sentiment = sum(weighted_scores) / total_weight
        else:
            avg_sentiment = None

        sentiment_scores.append(avg_sentiment)

    except Exception as e:
        print(f"Error for '{title}': {e}")
        sentiment_scores.append(None)

    time.sleep(1)  # respect Reddit's rate limits

df["reddit_sentiment_weighted"] = sentiment_scores
df.to_csv("reddit_sentiment_scores.csv", index=False)


100%|██████████| 954/954 [2:17:48<00:00,  8.67s/it]    


In [29]:
df

,Unnamed: 0,adult,id,original_language,overview,popularity,release_date,title,vote_average,vote_count,belongs_to_collection_boolean,budget,genres,origin_country,production_companies,revenue,runtime,num_spoken_languages,reddit_sentiment_weighted
0,0,False,278,en,Imprisoned in the 1940s for the double murder ...,34.6551,1994-09-23,The Shawshank Redemption,8.710,28243,False,25000000,"['Drama', 'Crime']",['US'],['Castle Rock Entertainment'],28341469,142,1,0.149323
1,2,False,238,en,"Spanning the years 1945 to 1955, a chronicle o...",50.6029,1972-03-14,The Godfather,8.686,21408,True,6000000,"['Drama', 'Crime']",['US'],"['Paramount Pictures', 'Alfran Productions']",245066411,175,3,0.027059
2,4,False,240,en,In the continuing saga of the Corleone crime f...,16.7806,1974-12-20,The Godfather Part II,8.571,12930,True,13000000,"['Drama', 'Crime']",['US'],"['Paramount Pictures', 'The Coppola Company', ...",102600000,202,4,0.247636
3,5,False,424,en,The true story of how businessman Oskar Schind...,24.8390,1993-12-15,Schindler's List,8.565,16409,False,22000000,"['Drama', 'History', 'War']",['US'],['Amblin Entertainment'],321365567,195,4,0.051268
4,7,False,389,en,The defense and the prosecution have rested an...,23.7899,1957-04-10,12 Angry Men,8.548,9109,False,397751,['Drama'],['US'],"['United Artists', 'Orion-Nova Productions']",4360000,97,1,0.500244
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
949,1087,False,267557,it,NaN,0.2815,2014-04-30,Un fidanzato per mia moglie,5.000,110,False,0,['Comedy'],['IT'],"['IBC MOvie', 'RAI']",0,97,1,NaN
950,1088,False,242088,en,Years after their successful restaurant review...,0.2821,2014-04-24,The Trip to Italy,6.325,243,True,0,"['Comedy', 'Drama']",['GB'],"['Revolution Films', 'Baby Cow Productions', '...",0,108,2,-0.253348
951,1089,False,43648,it,A poster worker must remove the poster of the ...,0.2821,1991-12-20,The Comics 2,5.700,217,True,0,['Comedy'],['IT'],"['Maura International Films', 'Penta Film', 'C...",0,87,1,NaN
952,1090,False,13296,tr,Failed magician Iskender decides to do a tour ...,0.2822,2006-10-20,The Magician,6.900,159,False,0,"['Comedy', 'Drama']",['TR'],['BKM Film'],0,122,1,0.225725


In [33]:
df.to_csv("MoviesWithRedditSentiment.csv", index = False)